# 03 · Experimentos estadísticos y nuevos datos

Este cuaderno continúa el EDA con modelos de lenguaje de palabras en Eráviri.
Estima probabilidades de secuencias; todavía no traduce al español. Las cifras
se obtienen ejecutando `python -m lexicon.experimental`. Los gráficos se regeneran
con `python -m lexicon.experimental_plots` en un entorno con Matplotlib.
Los modelos y sus pruebas numéricas usan solo la biblioteca estándar.


Ejecutar de arriba hacia abajo. Las tablas del texto corresponden a la ejecucion guardada; las celdas recalculan los resultados. Reinicia el kernel si antes cargaste paquetes de `.deps`.


In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                     if (p / 'ENUNCIADO.md').is_file() and (p / 'datos').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Abre este notebook dentro del proyecto.')
# No reutilizar binarios de .deps pertenecientes a otro Python.
sys.path[:] = [p for p in sys.path if Path(p or '.').resolve() != (PROJECT_ROOT / '.deps').resolve()]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Proyecto:', PROJECT_ROOT)
print('Python del kernel:', sys.version)
print('Interprete:', sys.executable)


Proyecto: C:\Users\aml\Desktop\GestionConocimiento
Python del kernel: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
Interprete: C:\Program Files\Python314\python.exe


La siguiente celda instala solo lo necesario para graficar en el kernel activo. El entrenamiento estadistico no necesita NumPy, pandas ni descargas de modelos.


In [ ]:
requirements_file = str(PROJECT_ROOT / "requirements-eda.txt")
%pip install -r "$requirements_file"


## 1. Qué llegó en fase 5 y cómo cambia el problema

| Archivo nuevo | Filas | Función experimental |
|---|---:|---|
| Pistas léxicas | 21 | Evidencia oficial para contrastar correspondencias |
| Comunicación inicial de Noelia | 18 | Expresiones conversacionales con traducción |
| Nuevos mensajes de fase 5 | 40 | Entradas nuevas sin referencia española |

Las 21 pistas son consistentes con entrenamiento: cinco sujetos, seis objetos,
cuatro verbos presentes, tres lugares, negación y dos sufijos. No constituyen
21 oraciones nuevas ni añaden raíces nuevas al vocabulario original. Para
comparar lugares se homogeneiza la representación: la pista dice `en la casa`
y el lexicón inducido guarda `la casa`. El prefijo `en` se añade al generar español.

Los sufijos `-ta` y `-ri` conservan su guion como metadato. No deben pasar por
tokenización general y convertirse en falsas palabras independientes `ta`/`ri`.
El contraste de estas pistas confirma la consistencia de una hipótesis; no
equivale a medir una mejora de traducción. Se exporta el soporte en entrenamiento
de cada pista a `contraste_pistas.csv`.

Las 18 expresiones conversacionales introducen 15 formas fuente nuevas. De sus
26 ocurrencias léxicas, 25 están fuera del vocabulario del LM principal: solo
`nelo` ya era conocido. Además tienen estructuras como `salia neri` o `mira?`
que no siguen necesariamente la plantilla sujeto–objeto–verbo. `noku` es una
respuesta aislada «No»; no debemos sustituir por ella la negación oracional `nu`.
El `?` se conserva en los originales y será importante para un futuro módulo
de expresiones; el LM actual utiliza la tokenización de palabras del proyecto.

Las expresiones son pares oficiales disponibles, no un conjunto de prueba
independiente. Memorizar sus 18 traducciones no demostraría generalización.
Hay traducciones con alternativas, como `Disculpa / Perdón`, que necesitarían
una política de referencias múltiples si se evaluaran automáticamente.

**Decisión:** mantener tres roles separados. No mezclar pistas, saludos y
oraciones declarativas para aumentar artificialmente el tamaño del corpus.
Un futuro traductor puede tener un módulo de expresiones conocidas y otro
composicional; debe distinguir respuestas conocidas de mensajes no cubiertos.


In [2]:
from lexicon.preprocessing import prepare, tokenize
from lexicon.experimental import (
    load_event, inspect_event, partition_monolingual, export_experiments,
    tokens_of, instantiate,
)
from lexicon.ngram import NGramLM

data, _ = prepare(PROJECT_ROOT)
event = load_event(PROJECT_ROOT)
audit = inspect_event(data, event)
print('Archivos nuevos:', audit['rows'])
print('Pistas consistentes:', audit['hints_consistent'])
print('Formas nuevas de conversacion:', audit['conversation_new_types'])
for row in event['conversation']:
    print(row['naviri'], '->', row['espanol'], '|', row['categoria'])


Archivos nuevos: {'conversation': 18, 'phase5': 40, 'hints': 21}
Pistas consistentes: 21
Formas nuevas de conversacion: ['areni', 'lumi', 'mali', 'mira', 'navira', 'naya', 'neri', 'noku', 'nori', 'pali', 'salia', 'selu', 'seni', 'vala', 'vali']
salia -> Hola | saludo
salia neri -> Buenos días | saludo
salia lumi -> Buenas tardes | saludo
salia nori -> Buenas noches | saludo
navira -> Adiós | despedida
navira seni -> Hasta pronto | despedida
mali -> Por favor | cortesía
areni -> Gracias | cortesía
areni vali -> Muchas gracias | cortesía
selu -> De nada | cortesía
naya -> Sí | respuesta
noku -> No | respuesta
seni? -> ¿Cómo estás? | conversación
seni vala -> Estoy bien | conversación
pali -> Disculpa / Perdón | cortesía
mira? -> ¿Puedes ayudarme? | petición
mira mali -> Ayúdame, por favor | petición
nelo mali -> Un mensaje, por favor | petición


## 2. Auditoría de los cuarenta mensajes nuevos

Los IDs son únicos y no hay fuentes idénticas a entrenamiento, validación,
monolingüe ni público original. No aparecen palabras nuevas respecto a los
420 pares; los 40 mensajes se pueden analizar con la hipótesis de estructura
obtenida de entrenamiento. Son compatibles con la gramática, no traducciones
ya verificadas.

En fase 5 hay 16 negaciones (40 %), 13 pasados (32,5 %), 13 futuros (32,5 %) y
20 lugares explícitos (50 %). En entrenamiento había 20,2 % de negaciones y
70,5 % de lugares. Es un cambio descriptivo relevante para probar robustez;
cuarenta ejemplos no bastan para inferir una distribución poblacional.

**Decisión:** no retocar reglas oración por oración ni inventar etiquetas. El
LM congelado puede asignar probabilidades a las fuentes, pero no medir exactitud
de traducción sin referencias. Fase 5 mantiene sus IDs `F5-...`; no se sustituye
el público original de 40 filas `NX-...` ni se sobrescribe su entrega.


In [3]:
print('Cruces fase 5:', audit['phase5_source_overlaps'])
print('OOV fase 5:', audit['phase5_oov_types'])
print('Estructuras reconocidas:', audit['phase5_parsed'])
print('Fenomenos:', audit['phase5_phenomena'])
for hint in audit['hint_checks']:
    print(hint['naviri'], '->', hint['espanol'], 'consistent:', hint['consistent_with_train'], 'support:', hint['train_support'])


Cruces fase 5: {'train': 0, 'validation': 0, 'monolingual': 0, 'public': 0}
OOV fase 5: []
Estructuras reconocidas: 40
Fenomenos: {'negative': 16, 'past': 13, 'future': 13, 'location': 20}
naku -> el niño consistent: True support: 38
mika -> la niña consistent: True support: 37
telu -> el maestro consistent: True support: 44
kemi -> la investigadora consistent: True support: 34
vonu -> el viajero consistent: True support: 44
pira -> el agua consistent: True support: 32
suni -> el libro consistent: True support: 35
moru -> el mapa consistent: True support: 41
dali -> la fruta consistent: True support: 42
nelo -> el mensaje consistent: True support: 33
zura -> la llave consistent: True support: 30
sema -> observa consistent: True support: 26
lomu -> busca consistent: True support: 26
dema -> lleva consistent: True support: 30
nemi -> recuerda consistent: True support: 18
hala -> en la casa consistent: True support: 45
miri -> en el bosque consistent: True support: 45
pavo -> en el labora

![04_evento_cambio_dominio](resultados/experimental_03/04_evento_cambio_dominio.png)


## 3. Preguntas e hipótesis experimentales

H1: añadir orden local (bigrama) reduce perplejidad respecto a frecuencias sin
contexto (unigrama), evaluando los mismos mensajes y vocabulario.

H2: el trigrama podría mejorar al bigrama por su contexto adicional, pero sufre
escasez de observaciones; esta hipótesis puede ser refutada por los datos.

H3: el monolingüe ofrece mejor estimación de contextos que solo 420 fuentes
bilingües. Hay que comparar corpus con el modelo fijo y considerar el tamaño
muestral como explicación, no atribuir toda diferencia al origen de los datos.

H4: las expresiones conversacionales quedan fuera del dominio del LM declarativo;
su OOV debe informarse junto con su perplejidad.

No se usan las pistas para modificar las probabilidades ni las traducciones de
validación para entrenar. La condición principal excluye las fuentes públicas
exactas del monolingüe; la exposición transductiva se analiza por separado.


## 4. División de datos antes de ajustar modelos

Se excluyen del monolingüe las fuentes exactas del público original y de fase 5.
La intersección real es de 28 mensajes públicos y cero de fase 5. Quedan 2.972
oraciones. Con una permutación de semilla `20261014` se crean:

| Partición del LM | Oraciones | Uso |
|---|---:|---|
| Entrenamiento | 2.377 | Ajustar vocabulario y conteos |
| Desarrollo | 297 | Elegir parámetros y familia del modelo principal |
| Prueba | 298 | Evaluación reservada; no elige parámetros |

El manifiesto `particiones_lm.csv` conserva IDs y textos para reconstruir la
división. Se verifica ausencia de fuentes compartidas entre las tres partes.
Esta división no reemplaza los CSV bilingües originales.

El vocabulario se ajusta solo con las 2.377 fuentes de entrenamiento del LM:
59 formas léxicas. Se añaden `<unk>` y `</s>` como posibles predicciones: 61
símbolos en total. `<s>` representa inicio, es contexto y nunca se predice.
Se comprueba que el entrenamiento bilingüe contiene el mismo inventario para
comparar corpus con un vocabulario idéntico. No se amplía con desarrollo o prueba.


In [4]:
splits, excluded = partition_monolingual(data, event)
print('Particiones:', {key: len(rows) for key, rows in splits.items()})
print('Fuentes publicas excluidas:', len(excluded))
print('Ejemplo de train del LM:', splits['train'][0])


Particiones: {'train': 2377, 'dev': 297, 'test': 298}
Fuentes publicas excluidas: 28
Ejemplo de train del LM: {'id': 'NX-1342', 'naviri': 'mika pira dema.', 'naviri_normalizado': 'mika pira dema.', 'naviri_tokens': ['mika', 'pira', 'dema']}


## 5. Qué calcula cada modelo

Unigrama: `P(palabra)`. Ejemplo: estima la probabilidad de `lomu` sin mirar
las palabras anteriores.

Bigrama: `P(palabra | anterior)`. Considera, por ejemplo, qué suele seguir a `nu`.

Trigrama: `P(palabra | dos anteriores)`. Tiene más contexto, pero reparte las
observaciones entre muchas más situaciones.

Se aplica suavizado aditivo:

```text
P(palabra | contexto) =
    (conteo(contexto, palabra) + alpha)
    / (conteo(contexto) + alpha × tamaño_del_vocabulario)
```

Alpha evita probabilidades cero. Demasiado suavizado hace las predicciones
más uniformes; demasiado poco penaliza mucho los contextos nuevos. Los
contextos que nunca se observaron tienen distribución uniforme en este modelo.

También se prueba interpolación: una media ponderada de probabilidades de
unigrama, bigrama y trigrama. Los pesos se eligen con desarrollo entre tres
alternativas, no se aprenden mirando la prueba.

Cada oración empieza con fronteras según el orden y termina en `</s>`. Nunca
se cuentan secuencias que crucen de una oración a otra. Las probabilidades
suman uno sobre los 61 símbolos de predicción para cualquier contexto.

`<unk>` recibe toda palabra no observada en entrenamiento. El suavizado reserva
probabilidad para él, incluso cuando no tiene ocurrencias de entrenamiento.
Como palabras distintas se colapsan a un solo símbolo, una perplejidad con muchos
OOV debe interpretarse con cautela y acompañarse de su tasa de desconocidos.


In [5]:
# Ejemplo didactico pequeno; no es el modelo final.
toy = NGramLM(order=2, alpha=1).fit([['a', 'b'], ['a', 'a']])
print('P(a | inicio) =', toy.probability('a'))
print('P(b | a) =', toy.probability('b', ['a']))
print('Suma de probabilidades =', sum(toy.probability(w, ['a']) for w in toy.vocabulary))


P(a | inicio) = 0.5
P(b | a) = 0.2857142857142857
Suma de probabilidades = 1.0


## 6. Diseño de comparación y búsqueda

Se prueban alpha `0.01, 0.05, 0.1, 0.5, 1.0`. El interpolado usa pesos
unigrama/bigrama/trigrama `(0.1,0.3,0.6)`, `(0.1,0.6,0.3)` y `(0.2,0.4,0.4)`.
Son 30 configuraciones por condición, 150 en total:

| Condición | Fuentes de entrenamiento | Propósito |
|---|---:|---|
| train_420 | 420 | Control con las fuentes bilingües originales |
| mono_clean | 2.377 | Experimento principal, sin exposición pública exacta |
| train_plus_mono | 2.797 | Añadir las 420 fuentes al monolingüe de entrenamiento |
| mono_transductive | 2.405 | Añadir las 28 fuentes públicas excluidas |
| mono_sample_420 | 420 | Control de tamaño: muestra solo de mono/train |

La muestra de 420 utiliza una semilla fija distinta (`20261015`) y no contiene
oraciones de desarrollo/prueba. Una sola muestra no permite estimar la
variabilidad entre muestras; este control es descriptivo.

Para cada familia y condición se eligen sus parámetros por menor perplejidad
de desarrollo. El modelo principal se elige entre las cuatro familias de
`mono_clean`, también usando exclusivamente desarrollo. No se reajusta con
desarrollo antes de medir prueba. Las otras condiciones responden preguntas
secundarias y no sustituyen al principal por obtener una cifra de prueba menor.

La perplejidad se calcula sobre todos los tokens predichos, incluyendo EOS:
`exp(suma de sorpresas / número total de tokens predichos)`. Se acumulan
logaritmos para evitar errores numéricos. No se promedian perplejidades de
oraciones, porque eso ponderaría incorrectamente sus distintas longitudes.


In [6]:
# Entrena las 150 configuraciones. Selecciona solo con desarrollo.
report = export_experiments(PROJECT_ROOT)
print('Configuraciones:', report['candidate_count'])
print('Modelo principal:', report['primary_model'])


Configuraciones: 150
Modelo principal: {'condition': 'mono_clean', 'family': 'bigram', 'order': 2, 'alpha': 0.01, 'weights': None, 'train_sentences': 2377, 'dev_perplexity': 8.456748899149371, 'dev_cross_entropy_bits': 3.0801031422418688, 'model_key': 'mono_clean__bigram', 'test': {'sentences': 298, 'predicted_tokens': 1446, 'lexical_tokens': 1148, 'oov_tokens': 0, 'oov_rate': 0.0, 'cross_entropy_nats': 2.2064139829010867, 'cross_entropy_bits': 3.183182511279464, 'perplexity': 9.083085820050146}}


## 7. Resultados del experimento principal

| Modelo | Alpha elegido | Pesos uni/bi/tri | Desarrollo | Prueba |
|---|---:|---|---:|---:|
| Unigrama | 0,01 | — | 35,95 | 36,75 |
| Bigrama | 0,01 | — | **8,46** | **9,08** |
| Trigrama | 0,05 | — | 12,14 | 13,77 |
| Interpolado | 0,01 | 0,1 / 0,6 / 0,3 | 9,39 | 10,00 |

Se favorece H1: el orden local aporta mucho frente a frecuencias aisladas.
H2 no se sostiene en esta comparación: el trigrama no mejora al bigrama. La
escasez de contextos es una explicación compatible con el EDA, no una prueba
causal aislada. La interpolación recupera parte del desempeño del trigrama,
pero sigue detrás del bigrama en desarrollo.

**Decisión:** congelar como LM principal el bigrama aditivo con alpha 0,01.
Es la mejor opción del espacio explorado, no un óptimo universal. Su alpha
está en el borde inferior de la grilla; probar valores menores sería un nuevo
experimento de desarrollo y no debe justificarse mirando la prueba.


In [7]:
for row in report['selected']:
    if row['condition'] == 'mono_clean':
        print(row['family'], 'alpha:', row['alpha'], 'weights:', row['weights'], 'dev:', round(row['dev_perplexity'], 4), 'test:', round(row['test']['perplexity'], 4))

from lexicon.experimental_plots import plot_experiments
figures = plot_experiments(PROJECT_ROOT)
print('Figuras generadas:', figures)


unigram alpha: 0.01 weights: None dev: 35.9479 test: 36.7468
bigram alpha: 0.01 weights: None dev: 8.4567 test: 9.0831
trigram alpha: 0.05 weights: None dev: 12.1436 test: 13.7671
interpolated alpha: 0.01 weights: (0.1, 0.6, 0.3) dev: 9.3858 test: 10.0053
Figuras generadas: ['01_modelos_dev_test', '02_suavizado_dev', '03_corpus_controlado', '04_evento_cambio_dominio', '05_fase5_sorpresa']


![01_modelos_dev_test](resultados/experimental_03/01_modelos_dev_test.png)


![02_suavizado_dev](resultados/experimental_03/02_suavizado_dev.png)


## 8. Aporte del corpus y control de factores

Comparar cada modelo con su mejor alpha cambiaría corpus e hiperparámetro a
la vez. Por eso se exporta además una comparación con **bigrama, alpha 0,01 y
vocabulario idénticos**:

| Corpus | Perplejidad de desarrollo | Perplejidad de prueba |
|---|---:|---:|
| 420 fuentes bilingües | 13,06 | 13,70 |
| 2.377 fuentes monolingües limpias | 8,46 | 9,08 |
| Bilingüe + monolingüe | 8,46 | 8,99 |
| Monolingüe + 28 entradas públicas | 8,46 | 9,08 |
| Muestra monolingüe de 420 | 11,58 | 12,17 |

El monolingüe resulta útil en este experimento. La comparación con una muestra
de 420 muestra que el volumen de datos también es un factor relevante. No se
afirma que un corpus sea intrínsecamente mejor a partir de una sola muestra.
Añadir las 420 fuentes al mono no mejora desarrollo; no se cambia de ganador
por su prueba ligeramente menor. Las diferencias pequeñas no se presentan
como mejoras estadísticamente significativas.


In [8]:
for row in report['fixed_bigram_ablation']:
    print(row['condition'], 'n:', row['train_sentences'], 'dev:', round(row['dev_perplexity'], 4), 'test:', round(row['test_perplexity'], 4))


train_420 n: 420 dev: 13.0558 test: 13.7029
mono_clean n: 2377 dev: 8.4567 test: 9.0831
train_plus_mono n: 2797 dev: 8.465 test: 8.9895
mono_transductive n: 2405 dev: 8.4583 test: 9.0795
mono_sample_420 n: 420 dev: 11.5799 test: 12.1695


![03_corpus_controlado](resultados/experimental_03/03_corpus_controlado.png)


## 9. Diagnóstico de los archivos externos con el LM congelado

| Fuentes evaluadas | Oraciones | Perplejidad | OOV léxico |
|---|---:|---:|---:|
| Validación bilingüe, solo Eráviri | 120 | 8,82 | 0 % |
| Público original | 40 | 8,99 | 0 % |
| Fase 5 | 40 | 10,54 | 0 % |
| Expresiones conversacionales | 18 | 2.050,19 | 96,15 % |

Estas evaluaciones no eligen el modelo. La fase 5 tiene algo más de sorpresa
para este LM, pero eso no demuestra errores de traducción. Las traducciones de validación no se leen
para ajustar el LM; solo se usan las fuentes.

La conversación tiene otro dominio y casi todo su vocabulario es desconocido.
Su cifra no es un ranking justo de calidad de traducción frente a los otros
archivos. Un LM entrenado en saludos podría responder a otra pregunta, pero
dieciocho expresiones no ofrecen aquí una base robusta de entrenamiento y
evaluación independientes.

`scores_phase5.csv` identifica mensajes de mayor sorpresa media por token.
Es una ayuda para inspección lingüística. No es una puntuación calibrada de
confianza, ni sirve para decidir automáticamente qué traducción es correcta.


In [9]:
for name, score in report['external_diagnostics'].items():
    print(name, 'PPL:', round(score['perplexity'], 3), 'OOV:', f"{100 * score['oov_rate']:.2f}%")

vocabulary = {w for sent in tokens_of(splits['train']) for w in sent}
lm = NGramLM.load(PROJECT_ROOT / 'resultados' / 'experimental_03' / 'modelo_principal.json')
import math
ranked = sorted(event['phase5'], key=lambda r: (lambda s: s['nll'] / s['predicted_tokens'])(lm.sentence_stats(tokenize(r['naviri']))), reverse=True)
print('Cinco fuentes de fase 5 con mayor sorpresa media:')
for row in ranked[:5]:
    score = lm.sentence_stats(tokenize(row['naviri']))
    print(row['id'], row['naviri'], 'bits/token:', round(score['nll'] / score['predicted_tokens'] / math.log(2), 3))


validation_source PPL: 8.817 OOV: 0.00%
public_source PPL: 8.989 OOV: 0.00%
phase5_source PPL: 10.537 OOV: 0.00%
conversation_shift PPL: 2050.189 OOV: 96.15%
Cinco fuentes de fase 5 con mayor sorpresa media:
F5-0012 vonu kava semari. bits/token: 5.42
F5-0015 lina tavi yelori. bits/token: 5.394
F5-0032 naku kava takuta. bits/token: 5.329
F5-0020 sari kava farota. bits/token: 5.273
F5-0004 telu moru taku. bits/token: 5.25


![05_fase5_sorpresa](resultados/experimental_03/05_fase5_sorpresa.png)


## 10. Qué queda listo y qué sigue pendiente

Queda implementado un LM de Eráviri con suavizado, interpolación, búsqueda de
parámetros, particiones reproducibles y evaluación separada. Los resultados
guardan versiones, semillas y hashes de todos los CSV. Los tests verifican
normalización de probabilidades, conteos manuales, fronteras, manejo de OOV,
consistencia de pistas y ausencia de fuentes compartidas en las particiones.

El próximo módulo debe aprender la relación Eráviri–español y compararse con
Jaccard. Un LM fuente no puede escoger por sí solo entre traducciones españolas
porque todas parten de la misma fuente. Si más adelante se usa un LM español
para ordenar candidatos, deberá demostrarse que mejora sin cambiar el sentido.

Para el traductor: conservar acción, participantes, lugar, negación y tiempo;
usar las pistas como evidencia declarada; tratar las expresiones oficiales
con una política explícita; evaluar exact match, chrF, fenómenos y combinaciones
nuevas. No se generó todavía `predicciones.csv` ni se inventaron referencias
para los 40 mensajes nuevos.


El archivo `modelo_principal.json` guarda los conteos y el vocabulario. Se carga con `NGramLM.load` sin volver a entrenar.


In [10]:
print('Resultados:', PROJECT_ROOT / 'resultados' / 'experimental_03')
print('Importante: perplejidad fuente no mide calidad de traduccion.')


Resultados: C:\Users\aml\Desktop\GestionConocimiento\resultados\experimental_03
Importante: perplejidad fuente no mide calidad de traduccion.
